[![Open In Colab](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/badge/open-in-colab.svg)](https://colab.research.google.com/github/crunchdao/quickstarters/blob/feat/structural-break-2/competitions/structural-break-streaming-data/quickstarters/dummy/dummy.ipynb)

![Banner](https://raw.githubusercontent.com/crunchdao/quickstarters/refs/heads/feat/structural-break-2/competitions/structural-break-streaming-data/assets/banner.webp)

# ADIA Lab Structural Break Streaming Data Challenge

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

# Setup

The first steps to get started are:
1. Get the setup command
2. Execute it in the cell below

### >> https://hub.crunchdao.io/competitions/structural-break-streaming-data/submit/notebook

![Reveal token](https://raw.githubusercontent.com/crunchdao/competitions/refs/heads/master/documentation/animations/reveal-token.gif)

In [ ]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

# Setup your local environment
!crunch setup --notebook structural-break hello --token aaaabbbbccccddddeeeeffff

In [ ]:
# Necessary for the staging environment
%env CRUNCH_COMPETITIONS_BRANCH=feat/structural-break-2
%env API_BASE_URL=https://api.hub.crunchdao.io/
%env WEB_BASE_URL=https://hub.crunchdao.io/

import crunch.store
crunch.store.load_from_env()

# Your model

## Setup

In [2]:
import os
from typing import Iterable, NamedTuple, NewType, Tuple

# Import your dependencies
import joblib
import pandas as pd
import scipy
import sklearn.metrics

In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Understanding the Data

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

In [ ]:
# Load the data simply
X_train, y_train, load_X_test = crunch_tools.load_data()

### Understanding `X_train`

The training data is structured as a pandas DataFrame with a MultiIndex:

**Index Levels:**
- `id`: Identifies the unique time series
- `time`: (arbitrary) The time step within each time series, which is regularly sampled

**Columns:**
- `value`: The values of the time series at each given time step
- `period`: whether you are in the first part of the time series (`0`), before the presumed break point, or in the second part (`1`), after the break point

In [5]:
X_train

value  period
id    time                  
0     0    -0.005564       0
      1     0.003705       0
      2     0.013164       0
      3     0.007151       0
      4    -0.009979       0
...              ...     ...
10000 2134  0.001137       1
      2135  0.003526       1
      2136  0.000687       1
      2137  0.001640       1
      2138  0.001074       1

[23715734 rows x 2 columns]

### Understanding `y_train`

This is a simple `pandas.Series` that tells if a time series id has a structural break, or not, from the presumed break point on.

**Index:**
- `id`: the ID of the time series

**Value:**
- `structural_breakpoint`: Boolean indicating whether a structural break occurred (`True`) or not (`False`)

In [6]:
y_train

id
0        False
1        False
2         True
3        False
4        False
         ...  
9996     False
9997     False
9998     False
9999     False
10000     True
Name: structural_breakpoint, Length: 10001, dtype: bool

### Understanding `X_test`

The test data is provided as a **`list` of `pandas.DataFrame`s** with the same format as [`X_train`](#understanding-X_test).

It is structured as a list to encourage processing records one by one, which will be mandatory in the `infer()` function.

In [21]:
X_test = load_X_test()

In [13]:
print("Number of datasets:", len(X_test))

Number of datasets: 101


Datasets are only iterable once, if you read it fully, make sure to store it somewhere as it will not be possible to read it again.

In [22]:
first_dataset = X_test[0]
first_dataset_df = pd.DataFrame(first_dataset)

if len(first_dataset_df) == 0:
    print("first_dataset_df is empty, you likely executed this cell twice")
    print("  re-execute the `X_test = load_X_test()` cell and then this one")
else:
    print("First dataset:")
    display(first_dataset_df)

First dataset:


,Index,value,period
0,"(10001, 0)",0.010753,0
1,"(10001, 1)",-0.031915,0
2,"(10001, 2)",-0.010989,0
3,"(10001, 3)",-0.011111,0
4,"(10001, 4)",0.011236,0
...,...,...,...
2774,"(10001, 2774)",-0.013937,1
2775,"(10001, 2775)",-0.015649,1
2776,"(10001, 2776)",-0.009744,1
2777,"(10001, 2777)",0.025375,1


## Strategy Implementation

Lorem ipsum dolor sit amet, consectetur adipiscing elit.

### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

The baseline implementation below doesn't require a pre-trained model, as it uses a statistical test that will be computed at inference time.

In [23]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    model_directory_path: str,
):
    # For our baseline t-test approach, we don't need to train a model
    # This is essentially an unsupervised approach calculated at inference time
    model = None

    # You could enhance this by training an actual model, for example:
    # 1. Extract features from before/after segments of each time series
    # 2. Train a classifier using these features and y_train labels
    # 3. Save the trained model

    joblib.dump(model, os.path.join(model_directory_path, 'model.joblib'))

### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

**Important workflow:**
1. Load your model;
2. Use the `yield` statement to signal readiness to the runner;
3. Process each dataset one by one within the for loop;
4. Each point you consume from the dataset will affect your score, the time between two iteration is also taken into account.
5. For each dataset, use `yield prediction` to return your prediction.

**Note:** The X_test object AND each datasets can only be iterated once!

In [ ]:
# @crunch/keep:on

DatasetId = NewType("DatasetId", int)
PointIndex = NewType("PointIndex", int)


# Not the real type, but used for clarity in the code
class Point(NamedTuple):
    Index: Tuple[DatasetId, PointIndex]  # named by Pandas
    value: float
    period: int

In [ ]:
def infer(
    X_test: Iterable[Iterable[Point]],
    model_directory_path: str,
):
    model = joblib.load(os.path.join(model_directory_path, 'model.joblib'))

    yield  # Mark as ready

    for dataset in X_test:
        result = 0.5  # Fallback to uncertain

        for point in dataset:
            # dataset_id, point_index = point.Index

            if point.period:  # select on breakpoint
                result = point.value
                break

        yield result

## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
crunch_tools.test(
    # Uncomment to disable the train
    # force_first_train=False,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

In [28]:
prediction = pd.read_parquet("prediction/prediction.parquet")
prediction

,prediction,duration
id,,
10001,0.021127,0 days 00:00:00.002842
10002,0.016566,0 days 00:00:00.002249
10003,0.047619,0 days 00:00:00.001817
10004,-0.007313,0 days 00:00:00.003604
10005,-0.006651,0 days 00:00:00.001706
...,...,...
10097,0.007823,0 days 00:00:00.002323
10098,-0.018035,0 days 00:00:00.002694
10099,0.014278,0 days 00:00:00.002950


You can also preview the duration for individual points stored in `prediction/prediction.duration.parquet`.

In [3]:
prediction_duration = pd.read_parquet("prediction/prediction.duration.parquet")
prediction_duration

,dataset_id,point_index,duration
0,10001,0,0 days 00:00:00.000003
1,10001,1,0 days 00:00:00.000002
2,10001,2,0 days 00:00:00.000001
3,10001,3,0 days 00:00:00.000001
4,10001,4,0 days 00:00:00.000001
...,...,...,...
182103,10101,1367,0 days 00:00:00.000001
182104,10101,1368,0 days 00:00:00.000001
182105,10101,1369,0 days 00:00:00.000001
182106,10101,1370,0 days 00:00:00.000001


### Local scoring

You can call the function that the system uses to estimate your score locally.

In [33]:
# Load the targets
target = pd.read_parquet("data/y_test.reduced.parquet")["structural_breakpoint"]

# Call the scoring function
sklearn.metrics.roc_auc_score(
    target,
    prediction["prediction"],
)

0.5014084507042254

# Submit your Notebook

To submit your work, you must:
1. Download your Notebook from Colab
2. Upload it to the platform
3. Create a run to validate it

Executing the cell below will take care of everything (only available on Google Colab), or show you how to submit manually.

In [ ]:
# @title  {"display-mode":"form", "form-width":"400px"}

# @markdown Describe your changes, then run the cell.
Message = "" # @param {"type":"string","placeholder":"Short description (optional)"}

# ---
# THIS METHOD IS ONLY POSSIBLE ON COLAB.
# RUNNING THIS CELL WILL PROMPT YOU TO USE THE OLD WAY OF SUBMITTING A NOTEBOOK.

crunch_tools.submit(
    message=Message,
)